# LeetCode 3136 — Valid Word bằng Python

Link đề: https://leetcode.com/problems/valid-word/ · Độ khó: **Easy** · Chủ đề: String

Skeleton LeetCode:

```python
class Solution:
    def isValid(self, word: str) -> bool:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một từ **hợp lệ** khi thỏa **bốn điều kiện cùng lúc**: dài **ít nhất `3` ký tự**; chỉ gồm **chữ cái và chữ số**; có **ít nhất một nguyên âm**; có **ít nhất một phụ âm**.
2. Nắm các công cụ chuỗi của Python: `len`, duyệt từng ký tự, `str.isalpha()`, `str.isdigit()`, `str.isalnum()`, `str.lower()`, toán tử `in`, và `any` / `all`.
3. Đi từ cách **một vòng lặp với các biến cờ (flag)** → **`all` / `any` với các hàm của `str`** → **tập hợp nguyên âm và phép giao `&`** → **biểu thức chính quy (regex) với `lookahead`** → **phân loại mỗi ký tự rồi đếm**.
4. Hiểu mấu chốt: **mỗi ký tự thuộc đúng một nhóm** — nguyên âm, phụ âm, chữ số, hoặc ký tự khác (`@`, `#`, `$`, ...); từ hợp lệ ⇔ **không có ký tự "khác"** và **độ dài `>= 3`** và **có nguyên âm** và **có phụ âm**. **Chữ số không phải phụ âm.**
5. Tránh các lỗi kinh điển: coi chữ số là phụ âm, quên chữ **hoa** (`A, E, I, O, U`), quên điều kiện độ dài, kiểm tra `isalpha` thay cho "chữ cái hoặc chữ số", và coi "chứa chuỗi con `aeiou`" là có nguyên âm.

## 2. Đề bài & ví dụ

Một từ được gọi là **hợp lệ** nếu thỏa **tất cả** các điều kiện sau:

- Có **ít nhất `3` ký tự**.
- Chỉ gồm **chữ số (`0`–`9`)** và **chữ cái tiếng Anh (chữ hoa hoặc chữ thường)**.
- Có **ít nhất một nguyên âm**.
- Có **ít nhất một phụ âm**.

**Nguyên âm** tiếng Anh là `'a'`, `'e'`, `'i'`, `'o'`, `'u'` (và dạng hoa của chúng). Một chữ cái tiếng Anh **không phải nguyên âm** thì là **phụ âm**. (Chữ số **không** phải nguyên âm cũng **không** phải phụ âm.)

Cho một chuỗi `word`, trả về `true` nếu `word` hợp lệ, ngược lại `false`.

**Ví dụ 1**

- Input: `word = "234Adas"`
- Output: `true`
- Giải thích: dài `7 >= 3`; chỉ gồm chữ số và chữ cái; có nguyên âm (`A`, `a`); có phụ âm (`d`, `s`).

**Ví dụ 2**

- Input: `word = "b3"`
- Output: `false`
- Giải thích: chỉ có `2` ký tự (`< 3`).

**Ví dụ 3**

- Input: `word = "a3$e"`
- Output: `false`
- Giải thích: chứa ký tự `'$'` không phải chữ số hay chữ cái. (Ngoài ra không có phụ âm.)

**Ràng buộc:** `1 <= word.length <= 20`; `word` chỉ gồm chữ cái tiếng Anh (hoa/thường), chữ số, và các ký tự `'@'`, `'#'`, `'$'`.

In [1]:
import random
import re
import string

examples = [
    ("234Adas", True),
    ("b3", False),
    ("a3$e", False),
]
for word, expected in examples:
    print(repr(word), "-> kỳ vọng", expected)

'234Adas' -> kỳ vọng True
'b3' -> kỳ vọng False
'a3$e' -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Duyệt chuỗi và `len`

`len(word)` là số ký tự; `for c in word` duyệt từng ký tự. Điều kiện "ít nhất `3` ký tự" là `len(word) >= 3`.

In [2]:
word = "234Adas"
print(len(word), len(word) >= 3)       # 7 True
print(list(word))                      # ['2', '3', '4', 'A', 'd', 'a', 's']

7 True
['2', '3', '4', 'A', 'd', 'a', 's']


### 3.2. Phân loại ký tự: `isalpha`, `isdigit`, `isalnum`

- `c.isalpha()`: `c` là **chữ cái**.
- `c.isdigit()`: `c` là **chữ số** `0`–`9`.
- `c.isalnum()`: `c` là chữ cái **hoặc** chữ số (`isalpha() or isdigit()` với ký tự ASCII).

Các ký tự `'@'`, `'#'`, `'$'` cho `False` ở cả ba hàm. Lưu ý: với văn bản không phải ASCII (chữ có dấu), `isalpha()`/`isalnum()` có thể trả `True` cho cả chữ cái không phải tiếng Anh; đề giới hạn ký tự nên không lo.

In [3]:
for c in "aB3$@":
    print(repr(c), c.isalpha(), c.isdigit(), c.isalnum())

'a' True False True
'B' True False True
'3' False True True
'$' False False False
'@' False False False


### 3.3. Chữ hoa và chữ thường: `lower()`

`c.lower()` đổi chữ hoa thành chữ thường (`'A'.lower() == 'a'`; chữ số và ký tự khác không đổi). Khi kiểm tra "là nguyên âm" ta nên **đổi về chữ thường trước** để chỉ cần liệt kê `'aeiou'` một lần.

In [4]:
print("A".lower(), "e".lower(), "3".lower(), "$".lower())    # a e 3 $
print("A".lower() in "aeiou", "B".lower() in "aeiou")        # True False

a e 3 $
True False


### 3.4. `in` với chuỗi/tập nguyên âm

`c in "aeiou"` hỏi "ký tự `c` có nằm trong chuỗi `aeiou`?" (với `c` là **một** ký tự). Cẩn thận: `"ae" in "aeiou"` là `True` (chuỗi con), nhưng không phải ý ta cần; chỉ dùng với **một** ký tự. Có thể dùng `set` cho nhanh: `VOWELS = set("aeiou")`.

In [5]:
VOWELS = set("aeiou")
print("a" in "aeiou", "b" in "aeiou", "A" in "aeiou")     # True False False (chưa lower!)
print("a" in VOWELS, "A".lower() in VOWELS)                 # True True
print("ae" in "aeiou")                                      # True (chuỗi con: coi chừng!)

True False False
True True
True


### 3.5. `any` và `all`

- `all(điều_kiện for c in word)`: **mọi** ký tự thỏa điều kiện (rỗng thì `True`).
- `any(điều_kiện for c in word)`: **ít nhất một** ký tự thỏa.

"Chỉ gồm chữ và số" là `all(c.isalnum() for c in word)`. "Có nguyên âm" là `any(c.lower() in VOWELS for c in word)`.

In [6]:
word = "234Adas"
print(all(c.isalnum() for c in word))                       # True
print(any(c.lower() in VOWELS for c in word))               # True
print(any(c.isalpha() and c.lower() not in VOWELS for c in word))   # True (có phụ âm)

True
True
True


### 3.6. Biểu thức chính quy (regex) cơ bản

`re.fullmatch(mẫu, chuỗi)` kiểm tra **toàn bộ** chuỗi có khớp mẫu không. Một số ký hiệu: `[A-Za-z0-9]` là một chữ cái hoặc chữ số; `{3,}` nghĩa là "lặp ít nhất `3` lần"; `(?=.*X)` (**lookahead**) khẳng định "từ vị trí hiện tại, ở đâu đó phía trước có `X`" mà **không tiêu thụ ký tự**. Ta sẽ dùng chúng ở Cách 4.

In [7]:
print(bool(re.fullmatch(r"[A-Za-z0-9]{3,}", "234Adas")))     # True
print(bool(re.fullmatch(r"[A-Za-z0-9]{3,}", "a3$e")))       # False
print(bool(re.match(r"(?=.*[aeiou])", "xyzae")))            # True (có nguyên âm ở đâu đó)

True
False
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Mỗi ký tự thuộc **đúng một** trong bốn nhóm:

| nhóm | ví dụ | có ảnh hưởng gì? |
|---|---|---|
| nguyên âm | `a e i o u A E I O U` | cho điều kiện "có nguyên âm" |
| phụ âm | các chữ cái còn lại | cho điều kiện "có phụ âm" |
| chữ số | `0`–`9` | hợp lệ nhưng **không** là nguyên âm/phụ âm |
| ký tự khác | `@ # $` ... | **làm từ không hợp lệ** |

Kiểm tra `"234Adas"`:

1. Độ dài `7 >= 3` ✔.
2. Mọi ký tự: `2 3 4` (số), `A d a s` (chữ) ⇒ không có ký tự khác ✔.
3. Nguyên âm: `A`, `a` ✔.
4. Phụ âm: `d`, `s` ✔.

Kiểm tra `"a3$e"`: có `'$'` thuộc nhóm "ký tự khác" ⇒ không hợp lệ ngay (và còn thiếu phụ âm).

In [8]:
def kind(c: str) -> str:
    if c.isdigit():
        return "số"
    if c.isalpha():
        return "nguyên âm" if c.lower() in "aeiou" else "phụ âm"
    return "khác"


for word in ["234Adas", "a3$e"]:
    print(word, [(c, kind(c)) for c in word])

234Adas [('2', 'số'), ('3', 'số'), ('4', 'số'), ('A', 'nguyên âm'), ('d', 'phụ âm'), ('a', 'nguyên âm'), ('s', 'phụ âm')]
a3$e [('a', 'nguyên âm'), ('3', 'số'), ('$', 'khác'), ('e', 'nguyên âm')]


## 5. Cách 1 — Brute Force / baseline: một vòng lặp với các biến cờ

**Ý tưởng.** Kiểm tra từng điều kiện theo đúng thứ tự trong đề:

1. Nếu `len(word) < 3` ⇒ `False`.
2. Duyệt từng ký tự `c`: nếu `c` là chữ cái, cập nhật cờ `has_vowel` hoặc `has_consonant`; nếu `c` là chữ số, bỏ qua; ngược lại (ký tự khác) ⇒ `False` ngay.
3. Cuối cùng trả `has_vowel and has_consonant`.

In [9]:
def is_valid_flags(word: str) -> bool:
    if len(word) < 3:                               # điều kiện độ dài
        return False
    has_vowel = False
    has_consonant = False
    for c in word:
        if c.isalpha():
            if c.lower() in "aeiou":
                has_vowel = True                    # tìm thấy nguyên âm
            else:
                has_consonant = True                # tìm thấy phụ âm
        elif c.isdigit():
            continue                                # chữ số: hợp lệ nhưng không ảnh hưởng hai cờ
        else:
            return False                            # ký tự không hợp lệ ('@', '#', '$')
    return has_vowel and has_consonant


for word, expected in examples:
    print(repr(word), is_valid_flags(word), "(kỳ vọng", expected, ")")

'234Adas' True (kỳ vọng True )
'b3' False (kỳ vọng False )
'a3$e' False (kỳ vọng False )


**Độ phức tạp (Cách 1).** Gọi `n = len(word)`. Một vòng lặp, mỗi ký tự `O(1)` ⇒ **Time `O(n)`**, **Space `O(1)`**.

**Nhận xét:** đã tối ưu về độ phức tạp. Các cách sau chỉ viết **gọn hơn** (hoặc khác phong cách).

## 6. Ý tưởng thuật toán / bước đột phá

Bốn điều kiện tách thành **bốn câu hỏi độc lập**, mỗi câu trả lời được bằng một biểu thức:

- `len(word) >= 3`
- `all(c.isalnum() for c in word)` — mọi ký tự là chữ hoặc số
- `any(c.isalpha() and c.lower() in VOWELS for c in word)` — có nguyên âm
- `any(c.isalpha() and c.lower() not in VOWELS for c in word)` — có phụ âm

Nối bằng `and`. Chú ý **phụ âm** phải **là chữ cái** (`isalpha()`) **và** không phải nguyên âm; nếu chỉ viết `c.lower() not in VOWELS` thì chữ số và ký tự khác cũng bị tính là phụ âm — **sai**.

In [10]:
word = "a3$e"
VOWELS = set("aeiou")
print("đủ dài:", len(word) >= 3)
print("chỉ chữ/số:", all(c.isalnum() for c in word))
print("có nguyên âm:", any(c.isalpha() and c.lower() in VOWELS for c in word))
print("có phụ âm:", any(c.isalpha() and c.lower() not in VOWELS for c in word))

đủ dài: True
chỉ chữ/số: False
có nguyên âm: True
có phụ âm: False


## 7. Các cách tiếp theo

### Cách 2 — `all` / `any` cho bốn điều kiện

**Giải thích.** Mỗi điều kiện một dòng, rồi `and` lại. Dễ đọc như đề bài, dễ kiểm tra từng điều kiện riêng.

In [11]:
VOWELS = set("aeiou")


def is_valid_allany(word: str) -> bool:
    return (
        len(word) >= 3
        and all(c.isalnum() for c in word)
        and any(c.isalpha() and c.lower() in VOWELS for c in word)
        and any(c.isalpha() and c.lower() not in VOWELS for c in word)
    )


for word, expected in examples:
    print(repr(word), is_valid_allany(word), "(kỳ vọng", expected, ")")

'234Adas' True (kỳ vọng True )
'b3' False (kỳ vọng False )
'a3$e' False (kỳ vọng False )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
class SolutionAllAny:
    def isValid(self, word: str) -> bool:
        vowels = set("aeiou")
        return (
            len(word) >= 3
            and all(c.isalnum() for c in word)
            and any(c.isalpha() and c.lower() in vowels for c in word)
            and any(c.isalpha() and c.lower() not in vowels for c in word)
        )


print(SolutionAllAny().isValid("234Adas"))      # True

True


**Độ phức tạp Cách 2:** `3` lượt duyệt `O(n)` ⇒ **Time `O(n)`**, **Space `O(1)`**. (`and` có **đoản mạch**: nếu một điều kiện sai thì các điều kiện sau không được tính.)

### Cách 3 — Tập hợp nguyên âm và phép giao

**Giải thích.** Gom các ký tự (chữ thường) thành các tập: `letters = {c.lower() for c in word if c.isalpha()}`. Khi đó **có nguyên âm** ⇔ `letters & VOWELS` khác rỗng; **có phụ âm** ⇔ `letters - VOWELS` khác rỗng (phép hiệu tập). Điều kiện "chỉ chữ/số" vẫn dùng `isalnum`.

In [13]:
def is_valid_sets(word: str) -> bool:
    if len(word) < 3 or not all(c.isalnum() for c in word):
        return False
    letters = {c.lower() for c in word if c.isalpha()}      # tập các chữ cái (đã đổi về thường)
    return bool(letters & VOWELS) and bool(letters - VOWELS)   # có nguyên âm và có phụ âm


for word, expected in examples:
    print(repr(word), is_valid_sets(word), "(kỳ vọng", expected, ")")

'234Adas' True (kỳ vọng True )
'b3' False (kỳ vọng False )
'a3$e' False (kỳ vọng False )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (tập tối đa 26 chữ cái).

### Cách 4 — Biểu thức chính quy với lookahead

**Giải thích.** Mẫu `^(?=.*[aeiouAEIOU])(?=.*[b-df-hj-np-tv-zB-DF-HJ-NP-TV-Z])[A-Za-z0-9]{3,}$`:

- `(?=.*[aeiouAEIOU])`: ở đâu đó có nguyên âm;
- `(?=.*[b-df-hj-np-tv-zB-DF-HJ-NP-TV-Z])`: ở đâu đó có phụ âm (các khoảng liệt kê mọi chữ cái trừ `a, e, i, o, u`);
- `[A-Za-z0-9]{3,}`: toàn bộ từ là chữ/số và dài `>= 3`.

Gọn, nhưng khó đọc và khó nhớ; tốt khi bạn quen regex.

In [14]:
PATTERN = re.compile(
    r"(?=.*[aeiouAEIOU])"                                    # có nguyên âm
    r"(?=.*[b-df-hj-np-tv-zB-DF-HJ-NP-TV-Z])"               # có phụ âm
    r"[A-Za-z0-9]{3,}"                                       # chỉ chữ/số, dài >= 3
)


def is_valid_regex(word: str) -> bool:
    return PATTERN.fullmatch(word) is not None


for word, expected in examples:
    print(repr(word), is_valid_regex(word), "(kỳ vọng", expected, ")")

'234Adas' True (kỳ vọng True )
'b3' False (kỳ vọng False )
'a3$e' False (kỳ vọng False )


**Độ phức tạp Cách 4:** **Time `O(n)`** với `n <= 20` (lookahead quét tối đa vài lần), **Space `O(1)`**. (Cẩn thận: với regex tổng quát, lookahead lồng có thể tốn `O(n²)` — ở đây `n` nhỏ.)

### Cách 5 — Phân loại từng ký tự rồi đếm

**Giải thích.** Dùng hàm `kind(c)` ở §4 để gán mỗi ký tự một nhóm, rồi `Counter` đếm số ký tự mỗi nhóm. Hợp lệ ⇔ `len >= 3`, số "khác" bằng `0`, số nguyên âm `>= 1`, số phụ âm `>= 1`. Cách này tách **phân loại** và **kết luận** — dễ mở rộng nếu đề đổi luật.

In [15]:
from collections import Counter


def is_valid_counter(word: str) -> bool:
    counts = Counter(kind(c) for c in word)               # {"nguyên âm": ..., "phụ âm": ..., "số": ..., "khác": ...}
    return (
        len(word) >= 3
        and counts["khác"] == 0
        and counts["nguyên âm"] >= 1
        and counts["phụ âm"] >= 1
    )


for word, expected in examples:
    print(repr(word), is_valid_counter(word), "(kỳ vọng", expected, ")")

'234Adas' True (kỳ vọng True )
'b3' False (kỳ vọng False )
'a3$e' False (kỳ vọng False )


**Độ phức tạp Cách 5:** **Time `O(n)`**, **Space `O(1)`** (tối đa 4 nhóm). (Truy cập `counts["khác"]` trên `Counter` trả `0` nếu khóa chưa có — không báo lỗi.)

## 8. TRACE từng bước

Ta trace Cách 1 (cờ) trên `"234Adas"`, `"a3$e"` và `"aaa"` (thiếu phụ âm).

In [16]:
def trace_flags(word: str) -> bool:
    print(f"word = {word!r}, độ dài = {len(word)}")
    if len(word) < 3:
        print("  độ dài < 3 -> False")
        return False
    has_vowel = has_consonant = False
    for i, c in enumerate(word):
        if c.isalpha():
            if c.lower() in "aeiou":
                has_vowel = True
                note = "nguyên âm"
            else:
                has_consonant = True
                note = "phụ âm"
        elif c.isdigit():
            note = "chữ số (bỏ qua)"
        else:
            print(f"  i={i} c={c!r}: ký tự không hợp lệ -> False")
            return False
        print(f"  i={i} c={c!r}: {note} | has_vowel={has_vowel}, has_consonant={has_consonant}")
    result = has_vowel and has_consonant
    print("  ->", result)
    return result


assert trace_flags("234Adas") is True
assert trace_flags("a3$e") is False
assert trace_flags("aaa") is False

word = '234Adas', độ dài = 7
  i=0 c='2': chữ số (bỏ qua) | has_vowel=False, has_consonant=False
  i=1 c='3': chữ số (bỏ qua) | has_vowel=False, has_consonant=False
  i=2 c='4': chữ số (bỏ qua) | has_vowel=False, has_consonant=False
  i=3 c='A': nguyên âm | has_vowel=True, has_consonant=False
  i=4 c='d': phụ âm | has_vowel=True, has_consonant=True
  i=5 c='a': nguyên âm | has_vowel=True, has_consonant=True
  i=6 c='s': phụ âm | has_vowel=True, has_consonant=True
  -> True
word = 'a3$e', độ dài = 4
  i=0 c='a': nguyên âm | has_vowel=True, has_consonant=False
  i=1 c='3': chữ số (bỏ qua) | has_vowel=True, has_consonant=False
  i=2 c='$': ký tự không hợp lệ -> False
word = 'aaa', độ dài = 3
  i=0 c='a': nguyên âm | has_vowel=True, has_consonant=False
  i=1 c='a': nguyên âm | has_vowel=True, has_consonant=False
  i=2 c='a': nguyên âm | has_vowel=True, has_consonant=False
  -> False


Bảng trace Cách 1 với `"234Adas"`:

| i | ký tự | nhóm | has_vowel | has_consonant |
|---|---|---|---|---|
| 0 | `2` | số | F | F |
| 1 | `3` | số | F | F |
| 2 | `4` | số | F | F |
| 3 | `A` | nguyên âm | T | F |
| 4 | `d` | phụ âm | T | T |
| 5 | `a` | nguyên âm | T | T |
| 6 | `s` | phụ âm | T | T |

Cuối vòng `has_vowel and has_consonant = True` ⇒ hợp lệ.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 1 (một vòng lặp với cờ): rõ ràng, thoát sớm khi gặp ký tự không hợp lệ, `O(n)`.

In [17]:
class Solution:
    def isValid(self, word: str) -> bool:
        if len(word) < 3:                         # điều kiện 1: ít nhất 3 ký tự
            return False
        has_vowel = has_consonant = False
        for c in word:
            if c.isalpha():                       # chữ cái
                if c.lower() in "aeiou":
                    has_vowel = True
                else:
                    has_consonant = True
            elif not c.isdigit():                 # không phải chữ cái cũng không phải chữ số: ký tự '@', '#', '$'
                return False
        return has_vowel and has_consonant        # điều kiện 3 và 4


sol = Solution()
for word, expected in examples:
    print(repr(word), "->", sol.isValid(word))

'234Adas' -> True
'b3' -> False
'a3$e' -> False


## 10. Vì sao thuật toán đúng

- **Đủ bốn điều kiện:** độ dài được kiểm tra ngay đầu; "chỉ chữ cái hoặc chữ số" được kiểm tra bằng nhánh `elif not c.isdigit(): return False` (mọi ký tự không phải chữ cái và không phải chữ số đều bị loại); nguyên âm và phụ âm được ghi bằng hai cờ.
- **Nguyên âm / phụ âm đúng:** với một **chữ cái**, đổi về chữ thường rồi hỏi có thuộc `aeiou` không: có ⇒ nguyên âm, không ⇒ phụ âm. Chữ số **không** đi vào nhánh `isalpha()`, nên không bao giờ bị tính là phụ âm.
- **Chữ hoa:** `c.lower()` đảm bảo `A, E, I, O, U` cũng là nguyên âm.
- **Kết luận:** nếu vòng lặp kết thúc mà không gặp ký tự xấu, từ chỉ gồm chữ/số; hợp lệ ⇔ cả hai cờ đều bật.
- **Tương đương các cách khác:** Cách 2–5 diễn đạt đúng bốn điều kiện đó bằng `all/any`, tập hợp, regex và đếm nhóm — đã được đối chiếu vét cạn và ngẫu nhiên ở §12.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(word)` (`n <= 20`).

| Cách | Time | Space |
|---|---|---|
| 1. Một vòng lặp với cờ | `O(n)` | `O(1)` |
| 2. `all` / `any` | `O(n)` | `O(1)` |
| 3. Tập hợp nguyên âm | `O(n)` | `O(1)` |
| 4. Regex lookahead | `O(n)` (với `n` nhỏ) | `O(1)` |
| 5. Phân loại + `Counter` | `O(n)` | `O(1)` |

Mọi cách đều chạy tức thì với `n <= 20`.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [18]:
sol = Solution()
impls = {
    "flags": is_valid_flags,
    "allany": is_valid_allany,
    "sets": is_valid_sets,
    "regex": is_valid_regex,
    "counter": is_valid_counter,
    "leetcode": sol.isValid,
}


def check_all(word, expected, label):
    for key, fn in impls.items():
        actual = fn(word)
        assert actual == expected, (label, key, word, actual, expected)
    print("PASS", label)


for idx, (word, expected) in enumerate(examples, 1):
    check_all(word, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [19]:
check_all("a", False, "một ký tự")
check_all("ab", False, "hai ký tự")
check_all("abc", True, "đúng 3 ký tự: nguyên âm a, phụ âm b, c")
check_all("aaa", False, "toàn nguyên âm: thiếu phụ âm")
check_all("bcd", False, "toàn phụ âm: thiếu nguyên âm")
check_all("AEI", False, "toàn nguyên âm hoa")
check_all("BCD", False, "toàn phụ âm hoa")
check_all("e3b", True, "có chữ số, nguyên âm và phụ âm")
check_all("123", False, "toàn chữ số: thiếu cả nguyên âm lẫn phụ âm")
check_all("a12", False, "có nguyên âm, chữ số không là phụ âm")
check_all("b12", False, "có phụ âm, chữ số không là nguyên âm")
check_all("a1b", True, "nguyên âm, chữ số, phụ âm")
check_all("abc@", False, "có ký tự @")
check_all("ab#c", False, "có ký tự #")
check_all("ab$c", False, "có ký tự $")
check_all("aBc", True, "chữ hoa/thường lẫn lộn")
check_all("Uxy", True, "nguyên âm hoa U")
check_all("a" * 19 + "b", True, "n = 20, hợp lệ")
check_all("a" * 20, False, "n = 20, thiếu phụ âm")

PASS một ký tự
PASS hai ký tự
PASS đúng 3 ký tự: nguyên âm a, phụ âm b, c
PASS toàn nguyên âm: thiếu phụ âm
PASS toàn phụ âm: thiếu nguyên âm
PASS toàn nguyên âm hoa
PASS toàn phụ âm hoa
PASS có chữ số, nguyên âm và phụ âm
PASS toàn chữ số: thiếu cả nguyên âm lẫn phụ âm
PASS có nguyên âm, chữ số không là phụ âm
PASS có phụ âm, chữ số không là nguyên âm
PASS nguyên âm, chữ số, phụ âm
PASS có ký tự @
PASS có ký tự #
PASS có ký tự $
PASS chữ hoa/thường lẫn lộn
PASS nguyên âm hoa U
PASS n = 20, hợp lệ
PASS n = 20, thiếu phụ âm


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi phổ biến: (a) **coi chữ số là phụ âm**; (b) **quên chữ hoa**; (c) **bỏ điều kiện độ dài**.

In [20]:
def wrong_digit_consonant(word):
    if len(word) < 3 or not all(c.isalnum() for c in word):
        return False
    has_vowel = any(c.lower() in "aeiou" for c in word)
    has_consonant = any(c.lower() not in "aeiou" for c in word)     # SAI: chữ số cũng được tính phụ âm
    return has_vowel and has_consonant


def wrong_lowercase_only(word):
    if len(word) < 3 or not all(c.isalnum() for c in word):
        return False
    has_vowel = any(c in "aeiou" for c in word)                      # SAI: quên chữ hoa
    has_consonant = any(c.isalpha() and c not in "aeiou" for c in word)
    return has_vowel and has_consonant


def wrong_no_length(word):
    if not all(c.isalnum() for c in word):
        return False
    return (any(c.isalpha() and c.lower() in "aeiou" for c in word)
            and any(c.isalpha() and c.lower() not in "aeiou" for c in word))   # SAI: bỏ điều kiện độ dài


bad1, bad2, bad3 = "a12", "Abc", "ab"
print(wrong_digit_consonant(bad1), "!=", is_valid_flags(bad1))
print(wrong_lowercase_only(bad2), "!=", is_valid_flags(bad2))
print(wrong_no_length(bad3), "!=", is_valid_flags(bad3))
assert wrong_digit_consonant(bad1) != is_valid_flags(bad1)
assert wrong_lowercase_only(bad2) != is_valid_flags(bad2)
assert wrong_no_length(bad3) != is_valid_flags(bad3)

True != False
False != True
True != False


### 12.3. Đối chiếu ngẫu nhiên có seed và vét cạn

Hai cách kiểm chứng: (1) **vét cạn mọi chuỗi dài `<= 3`** trên bảng ký tự đại diện (`a`, `b`, `A`, `B`, `1`, `$`) — đủ để bao mọi nhóm; (2) **ngẫu nhiên** các chuỗi độ dài `1..20` trên bảng ký tự đầy đủ của đề. Mọi cách phải khớp với một tham chiếu độc lập dựa trực tiếp vào định nghĩa đề.

In [21]:
def reference(word):
    vowels = "aeiouAEIOU"
    letters_ok = all(c in string.ascii_letters + string.digits for c in word)
    has_vowel = any(c in vowels for c in word)
    has_consonant = any(c in string.ascii_letters and c not in vowels for c in word)
    return len(word) >= 3 and letters_ok and has_vowel and has_consonant


import itertools
checked = 0
for n in range(1, 5):
    for chars in itertools.product("abAB1$", repeat=n):
        word = "".join(chars)
        ref = reference(word)
        for key, fn in impls.items():
            assert fn(word) == ref, (key, word)
        checked += 1
print("PASS vét cạn", checked, "chuỗi nhỏ")

rng = random.Random(3136)
pool = string.ascii_letters + string.digits + "@#$"
true_count = 0
for trial in range(3000):
    mode = rng.choice(["full", "alnum", "letters", "digits_heavy"])
    n = rng.randint(1, 20)
    if mode == "full":
        alphabet = pool
    elif mode == "alnum":
        alphabet = string.ascii_letters + string.digits
    elif mode == "letters":
        alphabet = string.ascii_letters
    else:
        alphabet = string.digits * 4 + "aeb"
    word = "".join(rng.choice(alphabet) for _ in range(n))
    ref = reference(word)
    true_count += ref
    for key, fn in impls.items():
        assert fn(word) == ref, (key, word)
print("PASS 3000 chuỗi ngẫu nhiên; số ca True =", true_count)
assert true_count > 300

PASS vét cạn 1554 chuỗi nhỏ
PASS 3000 chuỗi ngẫu nhiên; số ca True = 1497


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Coi chữ số là phụ âm

In [22]:
def sai_so_la_phu_am(word):
    if len(word) < 3 or not all(c.isalnum() for c in word):
        return False
    has_vowel = any(c.lower() in "aeiou" for c in word)
    has_consonant = any(c.lower() not in "aeiou" for c in word)       # SAI: '1' không phải nguyên âm nên bị tính phụ âm
    return has_vowel and has_consonant


print("sai:", sai_so_la_phu_am("a12"), "| đúng:", Solution().isValid("a12"))

sai: True | đúng: False


**Vì sao sai:** "không phải nguyên âm" **không** có nghĩa là phụ âm: phụ âm phải là **chữ cái**. `'1'` không nguyên âm nhưng cũng không phụ âm, nên `"a12"` (có nguyên âm, không có phụ âm) không hợp lệ. **Cách sửa:** `c.isalpha() and c.lower() not in "aeiou"`.

### Lỗi 2: Quên chữ hoa

In [23]:
def sai_quen_chu_hoa(word):
    if len(word) < 3 or not all(c.isalnum() for c in word):
        return False
    has_vowel = any(c in "aeiou" for c in word)                         # SAI: 'A', 'E'... không khớp
    has_consonant = any(c.isalpha() and c not in "aeiou" for c in word)
    return has_vowel and has_consonant


print("sai:", sai_quen_chu_hoa("Abc"), "| đúng:", Solution().isValid("Abc"))
print("sai:", sai_quen_chu_hoa("AEx"), "| đúng:", Solution().isValid("AEx"))

sai: False | đúng: True
sai: False | đúng: True


**Vì sao sai:** `'A' in "aeiou"` là `False`; nên `"Abc"` mất nguyên âm; còn `'A'` bị tính là **phụ âm** (vì là chữ cái không thuộc `aeiou`), đẩy kết quả sai theo hướng khác. **Cách sửa:** `c.lower()` trước khi so, hoặc liệt kê cả `"aeiouAEIOU"`.

### Lỗi 3: Bỏ điều kiện độ dài

In [24]:
def sai_thieu_do_dai(word):
    has_vowel = has_consonant = False
    for c in word:
        if c.isalpha():
            if c.lower() in "aeiou":
                has_vowel = True
            else:
                has_consonant = True
        elif not c.isdigit():
            return False
    return has_vowel and has_consonant          # SAI: quên len(word) >= 3


print("sai:", sai_thieu_do_dai("ab"), "| đúng:", Solution().isValid("ab"))

sai: True | đúng: False


**Vì sao sai:** `"ab"` có nguyên âm, phụ âm, chỉ chữ cái, nhưng chỉ dài `2 < 3` ⇒ không hợp lệ. **Cách sửa:** `if len(word) < 3: return False` đầu hàm.

### Lỗi 4: Dùng `isalpha` thay cho "chữ cái hoặc chữ số"

In [25]:
def sai_chi_chu_cai(word):
    if len(word) < 3 or not word.isalpha():                  # SAI: chữ số bị loại
        return False
    has_vowel = any(c.lower() in "aeiou" for c in word)
    has_consonant = any(c.lower() not in "aeiou" for c in word)
    return has_vowel and has_consonant


print("sai:", sai_chi_chu_cai("234Adas"), "| đúng:", Solution().isValid("234Adas"))

sai: False | đúng: True


**Vì sao sai:** đề cho phép **chữ số** trong từ hợp lệ (ví dụ `"234Adas"`). `isalpha()` loại chữ số. **Cách sửa:** dùng `isalnum()` (chữ cái hoặc chữ số), hoặc `c.isalpha() or c.isdigit()`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Bài "**xác thực chuỗi**" (validate): liệt kê từng điều kiện của đề thành các câu hỏi độc lập, rồi nối bằng `and`.
- Phân loại ký tự: `isalpha`, `isdigit`, `isalnum`; luôn nhớ ký tự **khác** (dấu, ký hiệu) rơi vào nhánh còn lại.
- Phụ âm = chữ cái **và** không phải nguyên âm (không phải "không phải nguyên âm").
- So sánh không phân biệt hoa thường: `c.lower()` một lần.
- Mẫu giống các bài *Valid Palindrome (125)*, *Detect Capital (520)*, *Strong Password Checker (420)*.
- Kiểm chứng bằng **vét cạn chuỗi ngắn trên bảng ký tự đại diện**: nhỏ mà bao đủ mọi nhóm.

## 15. Biến thể của bài

1. **Đổi luật:** cần **ít nhất hai** nguyên âm, hoặc **đúng một chữ số**.
2. **Trả về lý do không hợp lệ** (chuỗi mô tả điều kiện nào sai).
3. **Mật khẩu hợp lệ** (có chữ hoa, chữ thường, số, ký hiệu).
4. **Đếm số từ hợp lệ trong một danh sách.**
5. **Vowels of All Substrings (2063)**, **Check if the Sentence Is Pangram (1832)**.

In [26]:
def why_invalid(word):
    if len(word) < 3:
        return "quá ngắn"
    if not all(c.isalnum() for c in word):
        return "có ký tự không phải chữ/số"
    if not any(c.isalpha() and c.lower() in "aeiou" for c in word):
        return "thiếu nguyên âm"
    if not any(c.isalpha() and c.lower() not in "aeiou" for c in word):
        return "thiếu phụ âm"
    return "hợp lệ"


def count_valid(words):
    return sum(Solution().isValid(w) for w in words)


print([why_invalid(w) for w in ["b3", "a3$e", "aaa", "bcd", "234Adas"]])
print(count_valid(["abc", "b3", "234Adas", "aaa"]))        # 2
assert [why_invalid(w) for w in ["b3", "a3$e", "aaa", "bcd", "234Adas"]] == [
    "quá ngắn", "có ký tự không phải chữ/số", "thiếu phụ âm", "thiếu nguyên âm", "hợp lệ"]
assert count_valid(["abc", "b3", "234Adas", "aaa"]) == 2

['quá ngắn', 'có ký tự không phải chữ/số', 'thiếu phụ âm', 'thiếu nguyên âm', 'hợp lệ']
2


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Một vòng với cờ | thoát sớm, hai cờ | `O(n)` | `O(1)` | **nên nộp**, dễ gỡ lỗi |
| 2. `all` / `any` | mỗi điều kiện một dòng | `O(n)` | `O(1)` | đọc giống đề bài |
| 3. Tập hợp | `letters & VOWELS`, `letters - VOWELS` | `O(n)` | `O(1)` | khi quen với phép toán tập |
| 4. Regex | lookahead | `O(n)` | `O(1)` | khi nhiều ràng buộc kiểu mẫu |
| 5. Phân loại + `Counter` | đếm số ký tự mỗi nhóm | `O(n)` | `O(1)` | khi luật có thể đổi |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `"7".isdigit()`, `"E".lower() in "aeiou"`, `"z".isalpha() and "z" not in "aeiou"`, `Solution().isValid("xyz")`, `Solution().isValid("a1b")`.

**Bài 2 (tính tay).** Với `word = "Ao9"`, `word = "Zzz1"`, `word = "ie@b"`: lập bảng nhóm của từng ký tự (nguyên âm / phụ âm / số / khác) và cho kết quả.

**Bài 3 (điền TODO).** Hoàn thành `is_valid_todo(word)` theo Cách 1.

In [27]:
def is_valid_todo(word):
    # TODO 1: nếu len(word) < 3 thì trả False
    has_vowel = has_consonant = False
    for c in word:
        # TODO 2: nếu c là chữ cái: lower() rồi xét nguyên âm hay phụ âm, bật cờ tương ứng
        # TODO 3: ngược lại, nếu c không phải chữ số: trả False
        pass
    # TODO 4: trả has_vowel and has_consonant
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `is_valid_one_line(word)` bằng **một biểu thức** `return ...` dùng `len`, `all`, `any` (Cách 2), nhưng định nghĩa `VOWELS` **bên trong** hàm.

**Bài 5 (biến thể / mini challenge).** (a) Viết `is_valid_strict(word)`: như bài gốc nhưng **cần ít nhất hai nguyên âm khác nhau**; (b) Viết `first_invalid_index(word)` trả về **chỉ số ký tự không hợp lệ đầu tiên** (không phải chữ/số) hoặc `-1` nếu không có.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `True`, `True`, `True`, `False` (toàn phụ âm), `True`. Bài 3: nhánh `isalpha`, nhánh `elif not c.isdigit()`.
- **Mức 2 (vừa):** Bài 2: `Ao9`: `A` nguyên âm, `o` nguyên âm, `9` số ⇒ thiếu phụ âm ⇒ `False`; `Zzz1`: `Z, z, z` phụ âm, `1` số ⇒ thiếu nguyên âm ⇒ `False`; `ie@b`: có `@` ⇒ `False`. Bài 4: `return len(word) >= 3 and all(...) and any(...) and any(...)`.
- **Mức 3 (sâu):** Bài 5(a): dùng `set(c.lower() for c in word if c.lower() in "aeiou")` và kiểm tra `len(...) >= 2`. 5(b): `for i, c in enumerate(word): if not c.isalnum(): return i`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert "7".isdigit() is True
assert ("E".lower() in "aeiou") is True
assert ("z".isalpha() and "z" not in "aeiou") is True
assert Solution().isValid("xyz") is False
assert Solution().isValid("a1b") is True

# Bài 2
assert [kind(c) for c in "Ao9"] == ["nguyên âm", "nguyên âm", "số"]
assert [kind(c) for c in "Zzz1"] == ["phụ âm", "phụ âm", "phụ âm", "số"]
assert [kind(c) for c in "ie@b"] == ["nguyên âm", "nguyên âm", "khác", "phụ âm"]
assert [Solution().isValid(w) for w in ["Ao9", "Zzz1", "ie@b"]] == [False, False, False]


# Bài 3
def is_valid_todo_ans(word):
    if len(word) < 3:
        return False
    has_vowel = has_consonant = False
    for c in word:
        if c.isalpha():
            if c.lower() in "aeiou":
                has_vowel = True
            else:
                has_consonant = True
        elif not c.isdigit():
            return False
    return has_vowel and has_consonant


for word, expected in examples:
    assert is_valid_todo_ans(word) == expected


# Bài 4
def is_valid_one_line(word):
    return (len(word) >= 3 and all(c.isalnum() for c in word)
            and any(c.isalpha() and c.lower() in "aeiou" for c in word)
            and any(c.isalpha() and c.lower() not in "aeiou" for c in word))


for word, expected in examples:
    assert is_valid_one_line(word) == expected


# Bài 5(a)
def is_valid_strict(word):
    vowels_seen = {c.lower() for c in word if c.lower() in "aeiou"}
    return Solution().isValid(word) and len(vowels_seen) >= 2


assert is_valid_strict("aeb") is True
assert is_valid_strict("aab") is False        # chỉ một nguyên âm khác nhau (a)
assert is_valid_strict("Aeb") is True


# Bài 5(b)
def first_invalid_index(word):
    for i, c in enumerate(word):
        if not c.isalnum():
            return i
    return -1


assert first_invalid_index("ab$c") == 2
assert first_invalid_index("abc") == -1
assert first_invalid_index("@") == 0
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xác thực chuỗi bằng một vòng lặp với cờ)

```python
class Solution:
    def isValid(self, word: str) -> bool:
        if len(word) < 3:
            return False
        has_vowel = has_consonant = False
        for c in word:
            if c.isalpha():
                if c.lower() in "aeiou":
                    has_vowel = True
                else:
                    has_consonant = True
            elif not c.isdigit():
                return False
        return has_vowel and has_consonant
```

**Checklist**

- [ ] Có kiểm tra `len(word) >= 3`?
- [ ] Mọi ký tự là chữ cái hoặc chữ số (`@`, `#`, `$` làm từ không hợp lệ)?
- [ ] Nguyên âm tính cả **chữ hoa** (`c.lower()` trước khi so với `"aeiou"`)?
- [ ] Phụ âm phải là **chữ cái** và không phải nguyên âm (chữ số **không** phải phụ âm)?
- [ ] Đã thử `"b3"`, `"a3$e"`, `"aaa"`, `"bcd"`, `"a12"`, `"234Adas"`?
- [ ] Skeleton đúng `class Solution` / `def isValid(self, word: str) -> bool`?